<a href="https://colab.research.google.com/github/mbudisic/AIE6/blob/main/09_Finetuning_Embeddings/Fine_tuning_Embedding_Models_for_RAG_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Embeddings for RAG on Specific Data

As we start our "fine-tuning" week, we'll start with the lowest hanging improvement one can do for RAG - which is:

Fine-tuning embeddings!

- 🤝 Breakout Room #1:
  - Task 1: Dependencies and Boilerplate
  - Task 2: Loading Data
  - Task 3: Constructing a Fine-tuning Dataset
  - Task 4: Fine-tuning `snowflake-arctic-embed-l`
  - Task 5: Evaluating our Retriever



#### Basic Overview of Fine-tuning Embeddings

In essence, what we want to do when we fine-tune our embedding models is very simple:

```
Move the embeddings for questions relating to a document
closer together with that document
```

We can think of fine-tuning our embedding models as follows:

1) We have some pair of text items that *should* be closer together
  - `Question`, `Document` pairs
  - EX: `Who drives the bus?`, `The bus was driven by Kyle, the Bus Driver`.

2) We use these pairs as labeled data to fine-tune our embedding model.

The process of training helps the model more accurately associate our questions with the correct documents.

##### ❓ Question #1:

Describe the nuance between using Q&D pairs to train the embedding model vs. inter-document pairs/related sentences.

What caveats does this approach have? Are there any special considerations for what kind of Q's we should use?


##### ❗ Answer #1:

Training on "related sentences" encourages the model to stick to the theme
while generating text. That's likely essential for training the foundation model
as it would be useful for any application.

In the context of fine-tuning or RAG, it may be more important in situations where
the queries themselves are more generic, e.g., "Summarize this paper".

Training on Q/D pairs aims at a subtly different task - connecting the
semantic context of the query with the appropriate spot in the document.
This makes it more likely that the response will address the question asked by
the user when that question itself contains some relation to a specific context
in the document database.

As for caveats, perhaps we are in danger of disconnecting pieces of context by accident.
Since an explicit query-document will be used both as a positive example
and a negative (against all other rows in the document database), I would guess
it is important to maintain a chunk-overlap (e.g. two sentences per chunk,
with one-sentence overlap) in order to maintain context connection.

There may be additional danger that the real user queries will be substantially
different than autogenerated queries, e.g., use acronyms or information that is
not contained in the context. This may require adding context documents that
would help with "translating" queries from user-language to document-language.

Finally, I am not so sure about how successful the use of distinct rows as
negative examples is. At the same, asking the question generator to generate
*negative* examples seems very challenging ahead of production.


## Task 1: Dependencies and Boilerplate

We'll set up our `nest_asyncio` so we can leverage async loops in our Notebook.

We'll also install the required libraries we'll be using today, and set up our OpenAI API key!

### Nest Asyncio

In [1]:
import nest_asyncio

nest_asyncio.apply()

### Install Dependencies

>> NOTE: You do not need to do these steps if you are running this notebook locally with `uv`.

In [2]:
!uv pip install -qU langchain_openai langchain_huggingface langchain_core langchain langchain_community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5

In [3]:
!uv pip install -qU faiss-cpu python-pptx==1.0.2 nltk==3.9.1 pymupdf dotenv beautifulsoup4 lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 17.2 MB/s eta 0:00:00


In [4]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()
from google.colab import userdata

def load_env_if_not_present(key_name, prompt_message):
  try:
    os.environ[key_name] = userdata.get(key_name)
  except userdata.SecretNotFoundError:
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)
  finally:
    print(f"{key_name} retrieved.")

load_env_if_not_present("OPENAI_API_KEY","Please enter your OpenAI API key!")


OPENAI_API_KEY retrieved.


### Provide OpenAI API Key

## Task 2: Loading Data

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [5]:
!mkdir data

In [6]:
import os

if not os.path.exists("data/2023_llms.html"):
    !curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html
else:
    print("File data/2023_llms.html already exists, skipping download.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31554    0 31554    0     0  88725      0 --:--:-- --:--:-- --:--:-- 88884


In [7]:
if not os.path.exists("data/2024_llms.html"):
    !curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html
else:
    print("File data/2024_llms.html already exists, skipping download.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70721    0 70721    0     0   500k      0 --:--:-- --:--:-- --:--:--  504k


In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import BSHTMLLoader

path = "data/"
text_loader = DirectoryLoader(path, glob="*.html", loader_cls=BSHTMLLoader)

Next, we'll set up a classic naive chunking strategy as we only care that the documents get parsed into chunks that we can generate synthetic questions about.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 750,
    chunk_overlap  = 20,
    length_function = len
)

Next we can load/split these documents as follows.

> NOTE: You may need to run this cell twice to get it to work.

In [10]:
training_documents = text_splitter.split_documents(text_loader.load())

In [11]:
len(training_documents)

102

Next, we're going to associate each of our chunks with a unique identifier.

In [12]:
import uuid
from typing import Set
id_set = set()

def unique_id(where:Set[str], prefix:str=""):
  """Create a unique ID with respect to the given set.

  (Optional) prefix is used to distinguish queries from documents"""
  id = prefix+str(uuid.uuid4())
  while id in where:
    id = prefix+str(uuid.uuid4())

  assert not(id == prefix)
  return id


for document in training_documents:
  doc_id = unique_id(id_set,"D-")
  id_set.add(doc_id)
  document.metadata["id"] = doc_id

Next, we'll simply use naive Python slicing to create a training, test, and validation set to prepare our data for the next step.

In [13]:
training_split_documents = training_documents[:len(training_documents) - 24]
validation_split_documents = training_documents[len(training_documents) - 24:102-12]
test_split_documents = training_documents[102-12:]

## Task 3: Constructing a Fine-tuning Dataset

Using the nodes we created above, we can finally start constructing a fine-tuning dataset utilizing OpenAI's `gpt-4.1-mini`

The basic idea here is straightforward enough:

1. We look at a document
2. We generate questions that could be answered by that node

This gives us a number of question/context pairs that we can use to fine-tune our Embeddings model.

In [14]:
from langchain_openai import ChatOpenAI

qa_chat_model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

We'll create a simple Question Generation prompt to query `gpt-4.1-mini` to generate Questions for each retrieved context.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

qa_prompt = """\
Given the following context, you must generate questions based on only the provided context.

You are to generate {n_questions} questions which should be provided in the following format:

1. QUESTION #1
2. QUESTION #2
3. QUESTION #3
...

Each question should be a single line, and should not include any other text.

Context:
{context}
"""

qa_prompt_template = ChatPromptTemplate.from_template(qa_prompt)

We'll create a simple chain to query the LLM!

In [16]:
question_generation_chain = qa_prompt_template | qa_chat_model

There's a lot going on in this function - let's take a deeper look:

1. First, we provide a list of documents and a number of questions
2. We, for each document in our list, generate `n_questions` of questions.
3. We then associate those questions and contexts via a `UUID`.

> NOTE: The reason we're doing this `UUID` association is for ease of use later in the notebook.

##### 🏗️ Activity #1:

We have:

- Lists of `Documents` with the `metadata` field `id`.

We need:

- An object with key `id`, which have values `str` questions.
- An object with key `question_id`, which have values `List(str)` which will be a list of associated `context_id`.

An Example:

question_object:
```python
{
'b4b95fb6-f827-4454-aa5b-20e62733f172': 'What types of accessible formats are available for persons with disabilities?',
'df58ee4f-714c-419e-8324-94e5870574e2': 'How do accessible formats benefit persons with disabilities?',
'505fce8b-0e56-48de-a251-61027e396918': 'What are some of the risks associated with the increasing capabilities of AI systems that generate synthetic content?',
'8ff0ab33-60dc-4fee-8958-91bfb686aca8': 'Why is it important for providers of AI systems to embed technical solutions for marking and detecting synthetic content?'
}
 ```

 context_object:
 ```python
{
'b4b95fb6-f827-4454-aa5b-20e62733f172': ['dd75bf94-75f3-4603-8e4b-5522f6925638'],
'df58ee4f-714c-419e-8324-94e5870574e2': ['dd75bf94-75f3-4603-8e4b-5522f6925638'],
'505fce8b-0e56-48de-a251-61027e396918': ['ffe3893f-688c-48e8-90bd-7a9feb953d90'],
'8ff0ab33-60dc-4fee-8958-91bfb686aca8': ['ffe3893f-688c-48e8-90bd-7a9feb953d90'],
}
 ```

 As you can see, a piece of context can be associated with more than 1 question.

 The task is to write the Python function(s) to accomplish this task.

 Your function signature is provided below, along with the desired return values.

 > NOTE: You can make any modifications that you desire - assuming that you have the correct input and outputs.

Let's first take a look at the format we receive from the LLM.

In [17]:
test = question_generation_chain.invoke({"n_questions" : 5, "context" : """
Language models (LMs) have become ubiquitous in both NLP research
and in commercial product offerings. As their commercial
importance has surged, the most powerful models have become
closed off, gated behind proprietary interfaces, with important
details of their training data, architectures, and development
undisclosed. Given the importance of these details in
scientifically studying these models, including their biases and
potential risks, we believe it is essential for the research
community to have access to powerful, truly open LMs. To this
end, we have built OLMo, a competitive, truly Open Language
Model, to enable the scientific study of language models. Unlike
most prior efforts that have only released model weights and
inference code, we release OLMo alongside open training data and
training and evaluation code. We hope this release will empower
the open research community and inspire a new wave of innovation.
"""})

test.pretty_print()

================================== Ai Message ==================================

1. What has caused the most powerful language models to become closed off and proprietary?  
2. Why is it important for the research community to have access to powerful, truly open language models?  
3. What distinguishes OLMo from most prior efforts in releasing language models?  
4. What components are released alongside OLMo to support scientific study?  
5. What is the intended impact of releasing OLMo to the open research community?


Now, we need to extract the questions from that format:

In [18]:
import regex as re
def question_extract(raw:str):
    pattern = re.compile(r'^\d+\.\s+(.+)$', re.MULTILINE)
    return [match.strip() for match in re.findall(pattern, raw)]

question_extract(test.content)


['What has caused the most powerful language models to become closed off and proprietary?',
 'Why is it important for the research community to have access to powerful, truly open language models?',
 'What distinguishes OLMo from most prior efforts in releasing language models?',
 'What components are released alongside OLMo to support scientific study?',
 'What is the intended impact of releasing OLMo to the open research community?']

Let's see what we get if the format is wrong --- for handling edge-cases.

In [19]:
question_extract("Something without questions")

[]

In [20]:
import tqdm
import asyncio
from langchain.schema import Document
from typing import List
from warnings import warn

async def create_questions(documents: List[Document], n_questions: int):
  """Generate questions based on documents stored in a list

  Args:
      documents: List of Document objects containing text content to generate questions from
      n_questions: Number of questions to generate per document

  Returns:
      Tuple containing:
          - Dictionary mapping question IDs to question text
          - Dictionary mapping document IDs to lists of question IDs relevant to that document
  """
  questions = {}
  relevant_context = {}

  # Asynchronously generate questions for each document
  ## THIS MADE A HUGE DIFFERENCE IN SPEED
  all_questions_raw = await asyncio.gather( *(
      question_generation_chain.ainvoke({'context': doc.page_content, 'n_questions': n_questions})
      for doc in documents )
    )

  # Process document-questionlist pairs
  for doc,questions_raw  in zip(documents, all_questions_raw):

    doc_id = doc.metadata["id"]

    # populate the questions database
    question_list = question_extract(questions_raw.content)

    if len(question_list) == 0:
      warn(f"LLM didn't generate any questions based on {doc_id}.")
      continue

    for question in question_list:
        question_id = unique_id(questions.keys(), prefix="Q-")
        questions[question_id] = question
        if question_id not in relevant_context.keys() or len(relevant_context[question_id]) == 0:
          relevant_context[question_id] = [doc_id]
        else:
          relevant_context[question_id].append(doc_id)

  return questions, relevant_context

### REMOVE `await` IF NOT USING ASYNC (HINT: Use `async`)

We'll use the function to generate training, validation, and test data.

In [21]:
test_questions, test_relevant_contexts = await create_questions(test_split_documents, 2)
test_split_documents

[Document(metadata={'source': 'data/2023_llms.html', 'title': 'Stuff we figured out about AI in 2023', 'id': 'D-7f3bc485-62db-4eb9-bf0f-a678d40baee9'}, page_content='A lot of people are excited about AI agents—an infuriatingly vague term that seems to be converging on “AI systems that can go away and act on your behalf”. We’ve been talking about them all year, but I’ve seen few if any examples of them running in production, despite lots of exciting prototypes.\nI think this is because of gullibility.\nCan we solve this? Honestly, I’m beginning to suspect that you can’t fully solve gullibility without achieving AGI. So it may be quite a while before those agent dreams can really start to come true!\nCode may be the best application\nOver the course of the year, it’s become increasingly clear that writing code is one of the things LLMs are most capable of.'),
 Document(metadata={'source': 'data/2023_llms.html', 'title': 'Stuff we figured out about AI in 2023', 'id': 'D-1131f574-f0f6-495b

In [22]:
test_questions

{'Q-13549a14-0dfd-419f-9ff9-52f990a5d484': 'What is the main reason given for the lack of AI agents running in production despite many prototypes?',
 'Q-3d72cc6b-7f91-44a5-a61d-19559dd35b09': 'Why might achieving AGI be necessary to fully solve the problem of gullibility in AI agents?',
 'Q-79a35c94-6f22-4bfb-9d0f-ffff5d921c90': 'Why are the grammar rules of programming languages like Python and JavaScript considered less complicated than those of natural languages?',
 'Q-07a260fa-6e6c-4bb6-83d3-b141241293e3': 'What is one of the great weaknesses of large language models (LLMs) mentioned in the context?',
 'Q-da0daa3a-85a6-4707-8291-3f7edc2f6690': 'How does the ability of ChatGPT Code Interpreter to execute and correct code impact the problem of hallucination in code generation?',
 'Q-8759bca3-8c43-43ab-93a0-01c46d1bccfc': "Why might software engineers feel threatened by ChatGPT's capability to write code?",
 'Q-d126f951-294d-4d39-bc34-28410b64aecc': 'How can software engineers leverag

In [23]:
test_relevant_contexts

{'Q-13549a14-0dfd-419f-9ff9-52f990a5d484': ['D-7f3bc485-62db-4eb9-bf0f-a678d40baee9'],
 'Q-3d72cc6b-7f91-44a5-a61d-19559dd35b09': ['D-7f3bc485-62db-4eb9-bf0f-a678d40baee9'],
 'Q-79a35c94-6f22-4bfb-9d0f-ffff5d921c90': ['D-1131f574-f0f6-495b-8a6b-e90e22f3c3ff'],
 'Q-07a260fa-6e6c-4bb6-83d3-b141241293e3': ['D-1131f574-f0f6-495b-8a6b-e90e22f3c3ff'],
 'Q-da0daa3a-85a6-4707-8291-3f7edc2f6690': ['D-60641d09-01bc-4655-895e-b69f44e65578'],
 'Q-8759bca3-8c43-43ab-93a0-01c46d1bccfc': ['D-60641d09-01bc-4655-895e-b69f44e65578'],
 'Q-d126f951-294d-4d39-bc34-28410b64aecc': ['D-20ef0e88-4314-4c7b-8541-d67c4bf450ff'],
 'Q-0676effc-dfd9-4ca1-bada-70bc815f29ca': ['D-20ef0e88-4314-4c7b-8541-d67c4bf450ff'],
 'Q-2527769b-547d-4c24-9e47-631a7dbee74e': ['D-3edf8258-b1d2-438b-b94d-d141724a797e'],
 'Q-8e785d58-f38c-480c-b874-0ea5219056b0': ['D-3edf8258-b1d2-438b-b94d-d141724a797e'],
 'Q-da5ee7b9-6d8f-426b-8bcf-d37a67163548': ['D-8e7fa511-aa3d-454b-9fa5-947ff7a13bd8'],
 'Q-3a680769-e99c-4bf5-9bd0-6de1135f4f44': 

In [24]:
val_questions, val_relevant_contexts = await create_questions(validation_split_documents, 2)

In [25]:
training_questions, training_relevant_contexts = await create_questions(training_split_documents, 2)

### Reformating and Saving Datasets

Now, we can save our datasets for later use!

In [26]:
import json

training_corpus = {train_item.metadata["id"] : train_item.page_content for train_item in training_split_documents}

train_dataset = {
    "questions" : training_questions,
    "relevant_contexts" : training_relevant_contexts,
    "corpus" : training_corpus
}

with open("training_dataset.jsonl", "w") as f:
  json.dump(train_dataset, f)

In [27]:
validation_corpus = {val_item.metadata["id"] : val_item.page_content for val_item in validation_split_documents}

validation_dataset = {
    "questions" : val_questions,
    "relevant_contexts" : val_relevant_contexts,
    "corpus" : validation_corpus
}

with open("validation_dataset.jsonl", "w") as f:
  json.dump(validation_dataset, f)

In [28]:
train_corpus = {test_item.metadata["id"] : test_item.page_content for test_item in test_split_documents}

test_dataset = {
    "questions" : test_questions,
    "relevant_contexts" : test_relevant_contexts,
    "corpus" : train_corpus
}

with open("test_dataset.jsonl", "w") as f:
  json.dump(test_dataset, f)

## Task 4: Fine-tuning `snowflake-arctic-embed-l`

Now that we have a dataset, let's grab a `sentence-transformers` Embeddings model!

We'll be using Snowflake's [`snowflake-arctic-embed-l`](https://huggingface.co/Snowflake/snowflake-arctic-embed-l) as a base embeddings model.

It is a well performing embeddings model by itself, but there's a lot of very specific domain terms and vocabulary in our courpus - so lets fine-tune it and see what that can do for us!

>> NOTE: Skip installing dependencies if you are running this notebook locally.

In [29]:
!uv pip install -qU sentence_transformers datasets pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.2.1 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 20.0.0 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
pylibcudf-cu12 25.2.1 requires py

In [30]:
from sentence_transformers import SentenceTransformer

model_id = "Snowflake/snowflake-arctic-embed-l"
model = SentenceTransformer(model_id)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/85.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

We'll grab some necessary imports from `sentence_transformers` and `torch`.

> NOTE: PyTorch (`torch`) is a popular machine learning library - while we don't go very deep into PyTorch it's an incredibly powerful and interesting library! Please read more about it [here](https://pytorch.org/tutorials/beginner/basics/intro.html)!

In [31]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sentence_transformers import InputExample

We're using a toy batch size here to reflect the limited number of examples we have.

> NOTE: It is typical to use a much larger batch size (~64+), hardware permitting.

In [32]:
BATCH_SIZE = 10

Let's move our dataset into the expected format for training.

In [33]:
corpus = train_dataset['corpus']
queries = train_dataset['questions']
relevant_context = train_dataset['relevant_contexts']

examples = []
for query_id, query in queries.items():
    doc_id = relevant_context[query_id][0]
    text = corpus[doc_id]
    example = InputExample(texts=[query, text])
    examples.append(example)

Now we can create a `torch` `DataLoader`!

In [34]:
loader = DataLoader(
    examples, batch_size=BATCH_SIZE
)

Next up, we'll prepare our loss function!

Loss is an important part of training, fine-tuning, and more. If you want a deep dive on loss - you can check out our [event on loss!](https://www.youtube.com/watch?v=iB8FWR9aD5Q&t=8s).

The core loss we're using today is called `MultipleNegativesRankingLoss` - you can find more information [here](https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/losses/MultipleNegativesRankingLoss.py).

This is "wrapped" in `MatryoshkaLoss`, which you can read the implementation of [here](https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/losses/MatryoshkaLoss.py).

In [35]:
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss

matryoshka_dimensions = [768, 512, 256, 128, 64]
inner_train_loss = MultipleNegativesRankingLoss(model)
train_loss = MatryoshkaLoss(
    model, inner_train_loss, matryoshka_dims=matryoshka_dimensions
)

##### 🏗️ Activity #2:

Both of these losses sound "cool", but what are they - exactly - under the hood?

Why are these losses specifically doing? Please write a short summary of each loss.

> NOTE: This is a course focused on AI Engineering and the application of AI - looking for a hint? Try pasting the code (linked above) into ChatGPT/Claude to write the summary!

##### ❗Activity #2 answer:

`MultipleNegativesRankingLoss` provides an optimization cost based on association of a query with
context documents. It reads the paired query-document from the training set
as the "positive" example, and all other query-document pairings in the training set as "negative"
examples. In principle, one could include additional explicit negative pairings
but we do not use that here.

This will in turn become the "distance" beetween the embedding vectors corresponding
to queries and context documents.

We can think of the corresponding optimization as a problem where we have
points in a space (queries and context documents) and we know their
prescribed distances. The goal for the embedding model is to find coordinates
for each point such that the prescribed distances are obeyed as close as possible.
Moreover, we are training a "machine" for generating the coordinates in such a way
that for these given points, the above is true.

`MatryoshkaLoss` poses the additional question: what number of coordinates
are we allowed to tune? It basically says: think of the given list of coordinates
as an ordered list, e.g., (1,2,4).

We are training a 4-dimensional coordinate representation, but we form
the total loss by constraining that a certain number of parameters is zero

- 1: `(*,0,0,0)`
- 2: `(*,*,0,0)`
- 4: `(*,*,*,*)`

This is done by constraining which elements of the vector can be varied
during the gradient calculation. The losses for all these subproblems are summed
(with optional weights) which gives the final loss.

If weights are skipped and the example problem above is given,
the change in the first element incurs a loss from all three subproblems,
for the second element for two subproblems. So the impact of getting the coordinates
right is basically 3:2:1:1, meaning getting the first coordinate "wrong"
is 3x more dangerous than getting the last two coordinates wrong.

Consequently, when we (after training) reduce the dimension of the embedding
from 4 -> 2, we are effectively setting the last two coordinates = 0 for embedding representations,
therefore intentionally making an error in the last two coordinates.
Since the embedding model was trained to be more tolerant to errors in
last two coordinates, that is, more likely to preserve the prescribed inner-loss
distances when the last two coordinates are wrong, we effectively
get an embedding into a nested set of spaces where reducing the dimension
to a prescribed subdimension by throwing out coordinates is done in an optimal way.


Now we can set-up our evaluator.

> NOTE: Due to the formatting of our dataset - this is all we have to do!

In [36]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

corpus = validation_dataset['corpus']
queries = validation_dataset['questions']
relevant_context = validation_dataset['relevant_contexts']

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_context)

We'll train this model for 5 epochs, though you could increase this number if we had a significant amount more data.

In [37]:
EPOCHS = 10

It's training time!

> NOTE: We're manually defining a warm-up period here - this is just to provide a smooth ramp into our training!

In [38]:
import wandb

load_env_if_not_present("WANDB_API_KEY","Please enter your WANDB API key!")
load_env_if_not_present("HF_TOKEN","Please enter your HF token!")

wandb.init(mode="online",
               # Set the wandb entity where your project will be logged (generally your team name).
    entity="budisicm-virginia-commonwealth-university",
    # Set the wandb project where this run will be logged.
    project="Finetuning Embedding Models"
    )
wandb.run.name = wandb.run.id
wandb.run.save()


WANDB_API_KEY retrieved.


wandb: Currently logged in as: budisicm (budisicm-virginia-commonwealth-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Calling wandb.run.save without any arguments is deprecated.Changes to attributes are automatically persisted.


True

> NOTE: You may not see direct improvement during the training cycles - this is absolutely expected. We will verify performance later in the notebook.

In [39]:
from huggingface_hub import notebook_login

notebook_login()

In [40]:
warmup_steps = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path='finetuned_arctic_ft',
    show_progress_bar=True,
    evaluator=evaluator,
    evaluation_steps=50
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
16,No log,No log,0.833333,1.000000,1.000000,1.000000,0.833333,0.333333,0.200000,0.100000,0.833333,1.000000,1.000000,1.000000,0.933033,0.909722,0.909722
32,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.969244,0.958333,0.958333
48,No log,No log,0.875000,1.000000,1.000000,1.000000,0.875000,0.333333,0.200000,0.100000,0.875000,1.000000,1.000000,1.000000,0.953866,0.937500,0.937500
50,No log,No log,0.875000,1.000000,1.000000,1.000000,0.875000,0.333333,0.200000,0.100000,0.875000,1.000000,1.000000,1.000000,0.953866,0.937500,0.937500
64,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.969244,0.958333,0.958333
80,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.969244,0.958333,0.958333
96,No log,No log,0.875000,1.000000,1.000000,1.000000,0.875000,0.333333,0.200000,0.100000,0.875000,1.000000,1.000000,1.000000,0.948411,0.930556,0.930556
100,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.963789,0.951389,0.951389
112,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.963789,0.951389,0.951389
128,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.963789,0.951389,0.951389


In [43]:
load_env_if_not_present("HF_USER","Please enter your HuggingFace username!")

HF_USER retrieved.


In [ ]:
import uuid

model.push_to_hub(f"{os.environ["HF_USER"]}/legal-ft-{uuid.uuid4()}")

## Task 5: Evaluating our Retriever

Now that we have fine-tuned our retriever - let's see if it's worthwhile!

We'll start with some basic imports.

In [44]:
import pandas as pd

from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.documents import Document

Now we'll define a function that will help us evaluate our retrieval process.

> NOTE: We're assuming 1 correct document in a "hit".

In [45]:
def evaluate_openai(
    dataset,
    embed_model,
    top_k=5,
    verbose=False,
):
  corpus = dataset['corpus']
  questions = dataset['questions']
  relevant_docs = dataset['relevant_contexts']
  documents = [Document(page_content=content, metadata={"id": doc_id}) for doc_id, content in corpus.items()]
  vectorstore = FAISS.from_documents(documents, embed_model)

  retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

  eval_results = []
  for id, question in tqdm.tqdm(questions.items()):
    retrieved_nodes = retriever.invoke(question)
    retrieved_ids = [node.metadata["id"] for node in retrieved_nodes]
    expected_id = relevant_docs[id][0]
    is_hit = expected_id in retrieved_ids
    eval_results.append({"id": id, "question": question, "expected_id": expected_id, "is_hit": is_hit})

  return eval_results

All that's left to do is evaluate, we'll evaluate our model against:

1. OpenAI's closed source `text-embedding-3-small`
2. The base non-fine-tuned version of `Snowflake/snowflake-arctic-embed-l`.

Let's see how it stacks up!

### `text-embedding-3-small`

In [46]:
te3_openai = OpenAIEmbeddings(model="text-embedding-3-small")
te3_results = evaluate_openai(test_dataset, te3_openai)

100%|██████████| 24/24 [00:14<00:00,  1.67it/s]


In [47]:
te3_results_df = pd.DataFrame(te3_results)

In [48]:
te3_hit_rate = te3_results_df["is_hit"].mean()
te3_hit_rate

np.float64(1.0)

### `Snowflake/snowflake-arctic-embed-l` (base)

In [49]:
from langchain_huggingface import HuggingFaceEmbeddings

huggingface_embeddings = HuggingFaceEmbeddings(model_name="Snowflake/snowflake-arctic-embed-l")
arctic_embed_m_results = evaluate_openai(test_dataset, huggingface_embeddings)

100%|██████████| 24/24 [00:00<00:00, 47.40it/s]


In [50]:
arctic_embed_m_results_df = pd.DataFrame(arctic_embed_m_results)

In [51]:
arctic_embed_m_hit_rate = arctic_embed_m_results_df["is_hit"].mean()
arctic_embed_m_hit_rate

np.float64(0.9583333333333334)

### `Snowflake/snowflake-arctic-embed-l` (fine-tuned)

In [52]:
finetune_embeddings = HuggingFaceEmbeddings(model_name="finetuned_arctic_ft")
finetune_results = evaluate_openai(test_dataset, finetune_embeddings)

Some weights of BertModel were not initialized from the model checkpoint at finetuned_arctic_ft and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 24/24 [00:00<00:00, 46.71it/s]


In [53]:
finetune_results_df = pd.DataFrame(finetune_results)

In [54]:
finetune_hit_rate = finetune_results_df["is_hit"].mean()
finetune_hit_rate

np.float64(1.0)

## Task 1: Vibe Checking the RAG Pipeline

We're going to use our RAG pipeline to vibe check on some common phrases now that we've modified it!

### Creating New Chunks

In order to try and evaluate our system more fairly, let's create new chunks that we will use to create our Vector Store.

In [55]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap  = 50,
    length_function = len
)

training_documents = text_splitter.split_documents(text_loader.load())

### Base Chain

We'll start by constructing our base chain, which will use the untrained retrieval model.

#### R - Retrieval

In [56]:
from langchain_community.vectorstores import FAISS

base_vectorstore = FAISS.from_documents(training_documents, huggingface_embeddings)
base_retriever = base_vectorstore.as_retriever(search_kwargs={"k": 6})

#### A - Augmented

In [57]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and a question, you must answer the question. If you do not know the answer, you must state that you do not know.

Context:
{context}

Question:
{question}

Answer:
"""

rag_prompt_template = ChatPromptTemplate.from_template(RAG_PROMPT)

#### G - Generation

In [58]:
rag_llm =  ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0
)

#### RAG - LCEL RAG Pipeline

In [59]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

base_rag_chain = (
    {"context": itemgetter("question") | base_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt_template | rag_llm | StrOutputParser(), "context": itemgetter("context")}
)

In [60]:
base_rag_chain.invoke({"question" : "What is an agent?"})["response"]

'Based on the provided context, an "agent" in the context of AI refers to systems that are often described as capable of acting on your behalf, such as travel agents or digital assistants. However, the term is highly vague and lacks a clear, universally accepted definition. The discussions highlight that many people use the term differently—some see agents as systems that can go and act independently, while others think of them as LLMs with access to tools that can be used in loops to solve problems. Overall, the concept of an agent remains somewhat ambiguous and is often associated with the idea of autonomy, but without a precise or consistent meaning.'

In [61]:
base_rag_chain.invoke({"question" : "Who has produced better models than GPT-3?"})["response"]

'Several organizations have produced models that are better than GPT-3, including Anthropic, Mistral, Google, Meta, EleutherAI, Stability AI, TII in Abu Dhabi (Falcon), Microsoft Research, xAI, Replit, and Baidu.'

In [62]:
base_rag_chain.invoke({"question" : "What is the laziest time of the year for AI?"})["response"]

'The provided context does not specify a particular time of year that is considered the "laziest" for AI.'

In [63]:
base_rag_chain.invoke({"question" : "What is the largest model that Simon has run on his phone?"})["response"]

'The provided context does not specify the name "Simon" or details about the largest model he has run on his phone. Therefore, I do not know.'

### Fine-tuned Embedding Model

Now let's rebuild our RAG chain with the Fine-tuned model - the only component we need to change is our `FAISS` vectorstore!

In [64]:
finetune_vectorstore = FAISS.from_documents(training_documents, finetune_embeddings)
finetune_retriever = finetune_vectorstore.as_retriever(search_kwargs={"k": 6})

In [65]:
finetune_rag_chain = (
    {"context": itemgetter("question") | finetune_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt_template | rag_llm | StrOutputParser(), "context": itemgetter("context")}
)

In [66]:
finetune_rag_chain.invoke({"question" : "What is an Agent?"})["response"]

'Based on the provided context, an "agent" in the context of AI and LLMs is a term that is used in various ways but lacks a clear, universally accepted definition. Some interpret agents as systems that can act on your behalf, like a travel agent, or as LLMs given access to tools to solve problems iteratively. However, the term remains vague and often conflated with concepts like autonomy. The overall sentiment is that true, reliable AI agents have not yet materialized, partly due to issues like gullibility and the difficulty of distinguishing truth from fiction.'

In [67]:
finetune_rag_chain.invoke({"question" : "Who has produced better models than GPT-3?"})["response"]

'Several organizations have produced models that are better than GPT-3, including Anthropic, Mistral, Google, Meta, EleutherAI, Stability AI, TII (Falcon), Microsoft Research, xAI, Replit, and Baidu.'

In [68]:
finetune_rag_chain.invoke({"question" : "What is the laziest time of the year for AI?"})["response"]

'I do not know.'

In [69]:
finetune_rag_chain.invoke({"question" : "What is the largest model that Simon has run on his phone?"})["response"]

'The largest model that Simon has run on his phone is Mistral 7B.'

#### ❓Question #2:

Which LCEL RAG Chain do you think answered the questions better, and why?

## Task 2: RAGAS Evaluation

It's great to have some idea of how our system is doing based on vibe-checks, but let's use RAGAS to provide more insight info. on how things are improving!

> NOTE: Please recreate *exactly* the RAGAS process we used to evaluate RAG, baselining with the default retriever, and then comparing the new retriever. The includes the Synthetic Data Generation steps.

In [70]:
### YOUR CODE HERE